In [ ]:
import os 
os.chdir('../..')

import numpy as np
import pandas as pd
import seaborn as sns
import tellurium as te
import re
import textwrap
import matplotlib.pyplot as plt
import itertools
import pickle

from mpl_toolkits.axes_grid1 import make_axes_locatable

from roadrunner import Config, Logger
Logger.disableLogging()
Config.setValue(Config.ROADRUNNER_DISABLE_PYTHON_DYNAMIC_PROPERTIES, True)
Config.setValue(Config.LOADSBMLOPTIONS_RECOMPILE, False) 
Config.setValue(Config.LLJIT_OPTIMIZATION_LEVEL, 4)
Config.setValue(Config.LLVM_SYMBOL_CACHE, True)
Config.setValue(Config.LOADSBMLOPTIONS_OPTIMIZE_GVN, True)
Config.setValue(Config.LOADSBMLOPTIONS_OPTIMIZE_CFG_SIMPLIFICATION, True)
Config.setValue(Config.LOADSBMLOPTIONS_OPTIMIZE_INSTRUCTION_COMBINING, True)
Config.setValue(Config.LOADSBMLOPTIONS_OPTIMIZE_DEAD_INST_ELIMINATION, True)
Config.setValue(Config.LOADSBMLOPTIONS_OPTIMIZE_DEAD_CODE_ELIMINATION, True)
Config.setValue(Config.LOADSBMLOPTIONS_OPTIMIZE_INSTRUCTION_SIMPLIFIER, True)
Config.setValue(Config.SIMULATEOPTIONS_COPY_RESULT, True)

In [ ]:
with open('all_labels_parameters_error.dat', 'rb') as file:
    y = pickle.load(file)

In [ ]:
rfit = te.loada('iMC057.txt')

## Dummy test #1: HCT increases Citrate

In [ ]:
d = 1/10

In [ ]:
passed_params = []  # to store info on which parameter sets pass

start_group = 1   # 0-based index → Group 2
start_set = 298   # 0-based index → Set 298

for i, param_group in enumerate(y):  # loop over all 9 groups
    if i < start_group:
        continue  # skip earlier groups

    keys = param_group[0]
    values_list = param_group[1]

    # Make one dict per parameter set in this group
    param_dicts = [dict(zip(keys, values)) for values in values_list]

    for j, t in enumerate(param_dicts):  # loop through the 3 parameter sets in this group
        if i == start_group and j < start_set:
            continue
        # --- assign parameters ---
        param_labels_dict = {}
        param_values_dict = {}

        for k, pid in enumerate(rfit.model.getGlobalParameterIds()):
            if pid in t:
                param_values_dict[pid] = t[pid]
                param_labels_dict[pid] = k

        d = 1/10
        param_values = param_values_dict.copy()

        # Test 1: no HCT
        param_values['v73'] = 0
        rfit.model.setGlobalParameterValues(
            [*param_labels_dict.values(), 2, 49, 50, 51],
            [*param_values.values(), d, 0, 0, 0]
        )
        rfit.reset()
        try:
            simresults = rfit.simulate(0, 60 * 4 * 60)
        except RuntimeError as e:
            print(f"RuntimeError for Group {i+1}, Set {j+1}: {e}")
            continue
        citrate_noHCT = simresults['[Citrate]'][-1]

        # Test 2: 1x HCT
        param_values['v73'] = 1 / (1e-7) / d
        rfit.model.setGlobalParameterValues(
            [*param_labels_dict.values(), 2, 49, 50, 51],
            [*param_values.values(), d, 0, 0, 0]
        )
        rfit.reset()
        try:
            simresults = rfit.simulate(0, 60 * 4 * 60)
        except RuntimeError as e:
            print(f"RuntimeError for Group {i+1}, Set {j+1}: {e}")
            continue
        citrate_1HCT = simresults['[Citrate]'][-1]

        # Test 3: 10x HCT
        param_values['v73'] = 10 / (1e-7) / d
        rfit.model.setGlobalParameterValues(
            [*param_labels_dict.values(), 2, 49, 50, 51],
            [*param_values.values(), d, 0, 0, 0]
        )
        rfit.reset()
        try:
            simresults = rfit.simulate(0, 60 * 4 * 60)
        except RuntimeError as e:
            print(f"RuntimeError for Group {i+1}, Set {j+1}: {e}")
            continue
        citrate_10HCT = simresults['[Citrate]'][-1]

        print(f"Group {i+1}, Set {j+1}:")
        print(citrate_noHCT, citrate_1HCT, citrate_10HCT)

        # --- evaluation ---
        if citrate_10HCT > citrate_1HCT > citrate_noHCT:
            print("✓ Passed")
            passed_params.append((i, j, t))  # store group, index, and dict
        else:
            print("✗ Failed")

print("\n--- Summary ---")
print(f"{len(passed_params)} parameter sets passed.")
for group_idx, set_idx, params in passed_params:
    print(f"Group {group_idx+1}, Set {set_idx+1}: {params}")


In [ ]:
with open("passed_params_continued.pkl", "wb") as f:
    pickle.dump(passed_params, f)

In [ ]:
# Load both pickle files
with open("passed_params.pkl", "rb") as f:
    params1 = pickle.load(f)

with open("passed_params_continued.pkl", "rb") as f:
    params2 = pickle.load(f)

# Combine them
combined = params1 + params2

# Remove duplicates (in case some overlap)
# We'll use (group_idx, set_idx) as the unique identifier
seen = set()
unique_combined = []
for g, s, p in combined:
    if (g, s) not in seen:
        unique_combined.append((g, s, p))
        seen.add((g, s))

print(f"Combined {len(params1)} + {len(params2)} → {len(unique_combined)} unique parameter sets")

# Save the merged version
with open("passed_params_combined.pkl", "wb") as f:
    pickle.dump(unique_combined, f)

## Dummy test #2: NADH consumption

In [ ]:
passed_params_2 = []  # to store info on which parameter sets pass

for n, (group_idx, set_idx, params_dict) in enumerate(unique_combined):
    print(f"\nTesting passed set {n+1} (Group {group_idx+1}, Set {set_idx+1})")

    # --- assign parameters ---
    param_labels_dict = {}
    param_values_dict = {}

    for k, pid in enumerate(rfit.model.getGlobalParameterIds()):
        if pid in params_dict:
            param_values_dict[pid] = params_dict[pid]
            param_labels_dict[pid] = k

    d = 1/10
    param_values = param_values_dict.copy()

    # Test: NADH
    param_values['v73'] = 0
    param_values['v44'] = 1 / 0.083 / d
    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 50, 51],
        [*param_values.values(), d, 0, 0, 0]
    )

    rfit.reset()
    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group {group_idx+1}, Set {set_idx+1}: {e}")
        continue

    NADH0 = simresults['[NADH]'][0]
    NADH4 = simresults['[NADH]'][-1]

    print(f'NADH before: {NADH0} and after: {NADH4}')

    # --- evaluation ---
    if NADH4 < NADH0:
        print("✓ Passed")
        passed_params_2.append((group_idx, set_idx, params_dict))
    else:
        print("✗ Failed")

print("\n--- Summary ---")
print(f"{len(passed_params_2)} parameter sets passed.")
for group_idx, set_idx, params in passed_params_2:
    print(f"Group {group_idx+1}, Set {set_idx+1}: {params}")


In [ ]:
save_path = os.path.expanduser("~/passed_params_test2.pkl")  # home directory
with open(save_path, "wb") as f:
    pickle.dump(passed_params_2, f)

print(f"Saved to {save_path}")

## Dummy test #3: Glycine from Serine

In [ ]:
passed_params_3 = []  # to store info on which parameter sets pass

for n, (group_idx, set_idx, params_dict) in enumerate(passed_params_2):
    print(f"\nTesting passed set {n+1} (Group {group_idx+1}, Set {set_idx+1})")

    # --- assign parameters ---
    param_labels_dict = {}
    param_values_dict = {}

    for k, pid in enumerate(rfit.model.getGlobalParameterIds()):
        if pid in params_dict:
            param_values_dict[pid] = params_dict[pid]
            param_labels_dict[pid] = k

    d = 1/10
    param_values = param_values_dict.copy()

    # Test: NADH
    param_values['v73'] = 0
    param_values['v38'] = 1 / 0.001 / d

    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 50, 51],
        [*param_values.values(), d, 0, 0, 0]
    )

    rfit.reset()
    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group {group_idx+1}, Set {set_idx+1}: {e}")
        continue

    Gly0 = simresults['[Glycine]'][0]
    Gly4 = simresults['[Glycine]'][-1]

    print(f'Glycine before: {Gly0} and after: {Gly4}')

    # --- evaluation ---
    if Gly0 < Gly4:
        print("✓ Passed")
        passed_params_3.append((group_idx, set_idx, params_dict))
    else:
        print("✗ Failed")

print("\n--- Summary ---")
print(f"{len(passed_params_3)} parameter sets passed.")
for group_idx, set_idx, params in passed_params_3:
    print(f"Group {group_idx+1}, Set {set_idx+1}: {params}")


## Dummy test #4: Regeneration of NADH

In [ ]:
passed_params_4 = []  # to store info on which parameter sets pass

for n, (group_idx, set_idx, params_dict) in enumerate(passed_params_3):
    print(f"\nTesting set {n+1} (Group {group_idx+1}, Set {set_idx+1})")

    # --- assign parameters ---
    param_labels_dict = {}
    param_values_dict = {}

    for k, pid in enumerate(rfit.model.getGlobalParameterIds()):
        if pid in params_dict:
            param_values_dict[pid] = params_dict[pid]
            param_labels_dict[pid] = k

    d = 1/200
    param_values = param_values_dict.copy()

    # Test: No Fdh
    param_values['v73'] = 0

    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 51, 53],
        [*param_values.values(), d, 0, 0, 0]
    )

    rfit.reset()

    preNAD = rfit.getValue('NAD')
    preFormate = rfit.getValue('Formate')

    param_values['v73'] = 0
    param_values['v43'] = (1 / 5.6 / d) + (preNAD / 5.6 / d)
    param_values['v74'] = (10 / 0.001 / d) + (preFormate / 0.001 / d)

    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 51, 53],
        [*param_values.values(), d, 0, 0, 0]
    )

    rfit.reset()
    print(rfit.getValue('NAD'))
    print(rfit.getValue('Formate'))
    print(rfit.getValue('hEC11719'))

    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group {group_idx+1}, Set {set_idx+1}: {e}")
        continue

    NADH4_neg = simresults['[NADH]'][-1]
    rfit.reset()

    # Test: Fdh Expressed!
    param_values['v73'] = 0

    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 51, 53],
        [*param_values.values(), d, 0, 0, 0]
    )

    rfit.reset()

    rel2 = rfit.rel2

    param_values['v73'] = 0
    param_values['v43'] = (1 / 5.6 / d) + (preNAD / 5.6 / d)
    param_values['v74'] = (10 / 0.001 / d) + (preFormate / 0.001 / d)

    hEC11719 = 0.005
    p_hEC11719 = hEC11719 / (0.01 * d * rel2)

    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 51, 53],
        [*param_values.values(), d, 0, p_hEC11719, 0]
    )

    rfit.reset()
    print(rfit.getValue('NAD'))
    print(rfit.getValue('Formate'))
    print(rfit.getValue('hEC11719'))

    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group {group_idx+1}, Set {set_idx+1}: {e}")
        continue

    NADH4_fdh = simresults['[NADH]'][-1]

    print(f'NADH -fdh: {NADH4_neg} and +fdh: {NADH4_fdh}')

    # --- evaluation ---
    if NADH4_neg < NADH4_fdh:
        print("✓ Passed")
        passed_params_4.append((group_idx, set_idx, params_dict))
    else:
        print("✗ Failed")

print("\n--- Summary ---")
print(f"{len(passed_params_4)} parameter sets passed.")
for group_idx, set_idx, params in passed_params_4:
    print(f"Group {group_idx+1}, Set {set_idx+1}: {params}")

## Dummy test #5: Pathway efficacy

In [ ]:
rfit.model.getGlobalParameterIds()[17]

In [ ]:
passed_params_5 = []  # to store info on which parameter sets pass

for n, (group_idx, set_idx, params_dict) in enumerate(passed_params_4):
    print(f"\nTesting set {n+1} (Group {group_idx+1}, Set {set_idx+1})")

    # --- assign parameters ---
    param_labels_dict = {}
    param_values_dict = {}

    for k, pid in enumerate(rfit.model.getGlobalParameterIds()):
        if pid in params_dict:
            param_values_dict[pid] = params_dict[pid]
            param_labels_dict[pid] = k

    d = 1/200
    param_values = param_values_dict.copy()

    # Test: No enzymes or HCT, but all cofactors
    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 51, 53],
        [*param_values.values(), d, 0, 0, 0]
    )
    rfit.reset()

    rel0 = rfit.rel0
    rel1 = rfit.rel1
    rel2 = rfit.rel2
    rel3 = rfit.rel3

    preNADH = rfit.getValue('NADH')
    preFormate = rfit.getValue('Formate')
    prePyruvate = rfit.getValue('Pyruvate')
    preATP = rfit.getValue('ATP')
    preAcCoA = rfit.getValue('AcetylCoA')
    preHCO3 = rfit.getValue('HCO3')
    preAcetate = rfit.getValue('Acetate')
    preMdh = rfit.getValue('eEC11137')

    param_values['v73'] = 0
    param_values['v44'] = (1 / 0.083 / d) + (preNADH / 0.083 / d)
    param_values['v74'] = (10 / 0.001 / d) + (preFormate / 0.001 / d)
    param_values['v53'] = (1 / 0.001 / d) + (prePyruvate / 0.001 / d)
    param_values['v6'] = (1 / 1.5 / d) + (preATP / 1.5 / d)
    param_values['v10'] = (1 / 0.61 / d) + (preAcCoA / 0.61 / d)
    param_values['v35'] = (10 / 0.015 / d) + (preHCO3 / 0.015 / d)
    param_values['v7'] = (2 / 69.3 / d) + (preAcetate / 69.3 / d)

    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 51, 53],
        [*param_values.values(), d, 0, 0, 0]
    )

    rfit.reset()

    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group {group_idx+1}, Set {set_idx+1}: {e}")
        continue

    Mal_neg = simresults['[SMalate]'][-1]
    rfit.reset()

    # Test: Fdh Expressed!
    param_values['v73'] = 0

    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 51, 53],
        [*param_values.values(), d, 0, 0, 0]
    )

    rfit.reset()

    param_values['v73'] = 10 / (1e-7) / d
    param_values['v44'] = (1 / 0.083 / d) + (preNADH / 0.083 / d)
    param_values['v74'] = (10 / 0.001 / d) + (preFormate / 0.001 / d)
    param_values['v53'] = (1 / 0.001 / d) + (prePyruvate / 0.001 / d)
    param_values['v6'] = (1 / 1.5 / d) + (preATP / 1.5 / d)
    param_values['v10'] = (1 / 0.61 / d) + (preAcCoA / 0.61 / d)
    param_values['v35'] = (10 / 0.015 / d) + (preHCO3 / 0.015 / d)
    param_values['v7'] = (2 / 69.3 / d) + (preAcetate / 69.3 / d)

    hEC11719 = 0.001 # fdh
    p_hEC11719 = (hEC11719 / (0.01 * d * rel2))

    hEC6411 = 0.005 # pyc
    p_hEC6411 = (hEC6411 / (0.01 * d * rel1))

    eEC11137 = 0.001 # mdh
    p_eEC11137 = (eEC11137 / (0.012601796 * d * rel0)) + (preMdh / (0.012601796 * d * rel2))

    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 17, 49, 51, 53],
        [*param_values.values(), d, p_eEC11137, p_hEC6411, p_hEC11719, 0]
    )

    rfit.reset()
    print(rfit.getValue('hEC11719'))
    print(rfit.getValue('hEC6411'))
    print(rfit.getValue('eEC11137'))

    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group {group_idx+1}, Set {set_idx+1}: {e}")
        continue

    Mal_pos = simresults['[SMalate]'][-1]
    rfit.reset()

    print(f'Malneg: {Mal_neg} and Mal_pos: {Mal_pos}')

    # --- evaluation ---
    if Mal_neg < Mal_pos:
        print("✓ Passed")
        passed_params_5.append((group_idx, set_idx, params_dict))
    else:
        print("✗ Failed")

print("\n--- Summary ---")
print(f"{len(passed_params_5)} parameter sets passed.")

In [ ]:
with open("filtering_parameter_sets/final_params.pkl", "wb") as f:
    pickle.dump(passed_params_5, f)

In [ ]:
passed_params_6 = []  # to store info on which parameter sets pass

for n, (group_idx, set_idx, params_dict) in enumerate(passed_params_5):
    print(f"\nTesting set {n+1} (Group {group_idx+1}, Set {set_idx+1})")

    # --- assign parameters ---
    param_labels_dict = {}
    param_values_dict = {}

    for k, pid in enumerate(rfit.model.getGlobalParameterIds()):
        if pid in params_dict:
            param_values_dict[pid] = params_dict[pid]
            param_labels_dict[pid] = k

    d = 1/10
    param_values = param_values_dict.copy()

    param_values['v73'] = 0

    # Test: No enzymes or HCT, but all cofactors
    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 51, 53],
        [*param_values.values(), d, 0, 0, 0]
    )
    rfit.reset()

    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group {group_idx+1}, Set {set_idx+1}: {e}")
        continue

    cisAconitate_neg = simresults['[cis_Aconitate]'][-1]
    rfit.reset()

    # Test: HCT
    param_values['v73'] = 10 / (1e-7) / d

    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 51, 53],
        [*param_values.values(), d, 0, 0, 0]
    )

    rfit.reset()

    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group {group_idx+1}, Set {set_idx+1}: {e}")
        continue

    cisAconitate_HCT = simresults['[cis_Aconitate]'][-1]
    rfit.reset()

    print(f'No HCT: {cisAconitate_neg} and 10 HCT: {cisAconitate_HCT}')

    # --- evaluation ---
    if cisAconitate_neg < cisAconitate_HCT:
        print("✓ Passed")
        passed_params_6.append((group_idx, set_idx, params_dict))
    else:
        print("✗ Failed")

print("\n--- Summary ---")
print(f"{len(passed_params_6)} parameter sets passed.")

In [ ]:
passed_params_7 = []  # to store info on which parameter sets pass

for n, (group_idx, set_idx, params_dict) in enumerate(passed_params_6):
    print(f"\nTesting set {n+1} (Group {group_idx+1}, Set {set_idx+1})")

    # --- assign parameters ---
    param_labels_dict = {}
    param_values_dict = {}

    for k, pid in enumerate(rfit.model.getGlobalParameterIds()):
        if pid in params_dict:
            param_values_dict[pid] = params_dict[pid]
            param_labels_dict[pid] = k

    d = 1/200
    param_values = param_values_dict.copy()

    # Test: No enzymes or HCT, but all cofactors
    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 51, 53],
        [*param_values.values(), d, 0, 0, 0]
    )
    rfit.reset()

    rel0 = rfit.rel0
    rel1 = rfit.rel1
    rel2 = rfit.rel2
    rel3 = rfit.rel3

    preNADH = rfit.getValue('NADH')
    preFormate = rfit.getValue('Formate')
    prePyruvate = rfit.getValue('Pyruvate')
    preATP = rfit.getValue('ATP')
    preAcCoA = rfit.getValue('AcetylCoA')
    preHCO3 = rfit.getValue('HCO3')
    preAcetate = rfit.getValue('Acetate')
    preMdh = rfit.getValue('eEC11137')

    hEC11719 = 0.001 # fdh
    p_hEC11719 = (hEC11719 / (0.01 * d * rel2))

    eEC11137 = 0.001 # mdh
    p_eEC11137 = (eEC11137 / (0.012601796 * d * rel0)) + (preMdh / (0.012601796 * d * rel2))

    param_values['v73'] = 0
    param_values['v44'] = (1 / 0.083 / d) + (preNADH / 0.083 / d)
    param_values['v74'] = (10 / 0.001 / d) + (preFormate / 0.001 / d)
    param_values['v53'] = (1 / 0.001 / d) + (prePyruvate / 0.001 / d)
    param_values['v6'] = (1 / 1.5 / d) + (preATP / 1.5 / d)
    param_values['v10'] = (1 / 0.61 / d) + (preAcCoA / 0.61 / d)
    param_values['v35'] = (10 / 0.015 / d) + (preHCO3 / 0.015 / d)
    param_values['v7'] = (2 / 69.3 / d) + (preAcetate / 69.3 / d)

    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 17, 49, 51, 53],
        [*param_values.values(), d, p_eEC11137, 0, p_hEC11719, 0]
    )

    rfit.reset()

    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group {group_idx+1}, Set {set_idx+1}: {e}")
        continue

    Cit_neg = simresults['[Citrate]'][-1]
    Succ_neg = simresults['[Succinate]'][-1]
    
    rfit.reset()

    # Test: Pyc Expressed

    param_values['v73'] = 0
    param_values['v44'] = (1 / 0.083 / d) + (preNADH / 0.083 / d)
    param_values['v74'] = (10 / 0.001 / d) + (preFormate / 0.001 / d)
    param_values['v53'] = (1 / 0.001 / d) + (prePyruvate / 0.001 / d)
    param_values['v6'] = (1 / 1.5 / d) + (preATP / 1.5 / d)
    param_values['v10'] = (1 / 0.61 / d) + (preAcCoA / 0.61 / d)
    param_values['v35'] = (10 / 0.015 / d) + (preHCO3 / 0.015 / d)
    param_values['v7'] = (2 / 69.3 / d) + (preAcetate / 69.3 / d)

    hEC11719 = 0.001 # fdh
    p_hEC11719 = (hEC11719 / (0.01 * d * rel2))

    hEC6411 = 0.005 # pyc
    p_hEC6411 = (hEC6411 / (0.01 * d * rel1))

    eEC11137 = 0.001 # mdh
    p_eEC11137 = (eEC11137 / (0.012601796 * d * rel0)) + (preMdh / (0.012601796 * d * rel2))

    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 17, 49, 51, 53],
        [*param_values.values(), d, p_eEC11137, p_hEC6411, p_hEC11719, 0]
    )

    rfit.reset()

    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group {group_idx+1}, Set {set_idx+1}: {e}")
        continue

    Cit_pyc = simresults['[Citrate]'][-1]
    Succ_pyc = simresults['[Succinate]'][-1]
    rfit.reset()

    print(f'Cit-pyc: {Cit_neg} Cit+pyc: {Cit_pyc}, Succ-pyc: {Succ_neg}, Succ+pyc{Succ_pyc}')

    # --- evaluation ---
    if Cit_neg < Cit_pyc and Succ_neg < Succ_pyc:
        print("✓ Passed")
        passed_params_7.append((group_idx, set_idx, params_dict))
    else:
        print("✗ Failed")

print("\n--- Summary ---")
print(f"{len(passed_params_7)} parameter sets passed.")

In [ ]:
sorted_params = []  # to store info on which parameter sets pass
diff_citrate = []

for n, (group_idx, set_idx, params_dict) in enumerate(passed_params_7):
    print(f"\nTesting set {n+1} (Group {group_idx+1}, Set {set_idx+1})")

    # --- assign parameters ---
    param_labels_dict = {}
    param_values_dict = {}

    for k, pid in enumerate(rfit.model.getGlobalParameterIds()):
        if pid in params_dict:
            param_values_dict[pid] = params_dict[pid]
            param_labels_dict[pid] = k

    d = 1/10
    param_values = param_values_dict.copy()

    # Test 1: no HCT
    param_values['v73'] = 0
    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 50, 51],
        [*param_values.values(), d, 0, 0, 0]
    )
    rfit.reset()
    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group {i+1}, Set {j+1}: {e}")
        continue
    citrate_noHCT = simresults['[Citrate]'][-1]

    # Test 2: 1x HCT
    param_values['v73'] = 10 / (1e-7) / d
    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 50, 51],
        [*param_values.values(), d, 0, 0, 0]
    )
    rfit.reset()
    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group {i+1}, Set {j+1}: {e}")
        continue
    citrate_10HCT = simresults['[Citrate]'][-1]

    diff_citrate.append(citrate_10HCT - citrate_noHCT)
    print(n)
    print(citrate_10HCT - citrate_noHCT)

In [ ]:
# Combine, sort, and unpack
sorted_pairs = sorted(zip(diff_citrate, passed_params_7), key=lambda x: x[0], reverse=True)

# Unzip back into separate lists (now both sorted)
diff_citrate_sorted, passed_params_7_sorted = zip(*sorted_pairs)

# Optionally convert back to lists (since zip() returns tuples)
diff_citrate_sorted = list(diff_citrate_sorted)
passed_params_7_sorted = list(passed_params_7_sorted)


In [ ]:
with open('filtering_parameter_sets/final_params.pkl', 'rb') as file:
    trainparams = pickle.load(file)

In [ ]:
passed_params = []  # to store info on which parameter sets pass

for n, (group_idx, set_idx, params_dict) in enumerate(trainparams):
    print(f"\nTesting set {n+1} (Group {group_idx+1}, Set {set_idx+1})")

    # --- assign parameters ---
    param_labels_dict = {}
    param_values_dict = {}

    for k, pid in enumerate(rfit.model.getGlobalParameterIds()):
        if pid in params_dict:
            param_values_dict[pid] = params_dict[pid]
            param_labels_dict[pid] = k

    d = 1/10
    param_values = param_values_dict.copy()

    # Test 1: no HCT
    param_values['v73'] = 0
    param_values['v53'] = (0 / 0.001 / d)
    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 50, 51],
        [*param_values.values(), d, 0, 0, 0]
    )
    rfit.reset()
    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group")
        continue
    citrate_noHCT = simresults['[Citrate]'][-1]

    # Test 2: 1x HCT
    param_values['v73'] = 1 / (1e-7) / d
    param_values['v53'] = (0 / 0.001 / d)
    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 50, 51],
        [*param_values.values(), d, 0, 0, 0]
    )
    rfit.reset()
    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group")
        continue
    citrate_1HCT = simresults['[Citrate]'][-1]

    # Test 3: 10x HCT
    param_values['v73'] = 10 / (1e-7) / d
    rfit.model.setGlobalParameterValues(
        [*param_labels_dict.values(), 2, 49, 50, 51],
        [*param_values.values(), d, 0, 0, 0]
    )
    rfit.reset()
    try:
        simresults = rfit.simulate(0, 60 * 4 * 60)
    except RuntimeError as e:
        print(f"RuntimeError for Group")
        continue
    citrate_10HCT = simresults['[Citrate]'][-1]

    print(citrate_noHCT, citrate_1HCT, citrate_10HCT)

    # --- evaluation ---
    if citrate_10HCT > citrate_1HCT > citrate_noHCT:
        print("✓ Passed")
        passed_params.append((group_idx, set_idx, params_dict))
    else:
        print("✗ Failed")